In [2]:
import random 
import numpy as np
import glob
import cv2
from sklearn.utils import shuffle
from matplotlib import pyplot as plt
from sklearn.cluster import KMeans
import joblib

pssm_glob = 'psiblast/data/pssm/*.pssm'

In [20]:
def read_matrix_from_pssm_file(fname):
    nrows=0
    with open(fname) as txt_file:    # Count rows.
        for line in txt_file:
            l = line.strip().split()
            if (len(l) == 44):
                nrows+=1;
    matrix = np.zeros((nrows,20), dtype="int8")
    i=0
    with open(fname) as txt_file: # Read PSSM.
        for line in txt_file:
            l = line.strip().split()
            if (len(l) == 44):
                values = np.array(list(map(int, l[2:22])))
                matrix[i,:] = values
                i+=1
    return matrix

# LOAD PSSM Matrices into protein_descriptors
protein_descritors = {}
for file in glob.glob(pssm_glob):
    pssm_matrix = read_matrix_from_pssm_file(file)
    prot_name = file.split("\\")[1].replace(".pssm", "")
    protein_descritors[prot_name] = pssm_matrix 
    
print("PSSM Matrixes read:")
print(len(protein_descritors))
#joblib.dump(protein_descritors,"protein_descritors.pkl", compress=3)

PSSM Matrixes read:
0


In [3]:
protein_descritors = joblib.load("protein_descritors.pkl")

In [4]:
# Create expected outputs array from sec_strucure file DSSP
dataset_file = open("sec_structs.txt", "r")
protein_secundary_structure = {}
protein_aa = {}
x = 0
name = ""
for line in dataset_file:
    if (x%4==0):
        prot_name = line.strip()
    elif(x%4==1):
        protein_aa[prot_name] = line.strip()
    elif(x%4==3):
        protein_secundary_structure[prot_name] = line.strip()
    x+=1

proteins = list(protein_secundary_structure.keys())
print(len(proteins))

1461


In [5]:
# for each proteins, for each amino extracts PSSM window with AA centered.
from scipy.ndimage import shift

X_protein_features = {}

protein_index = 0
for protein in proteins:
    pssm = protein_descritors[protein]
    protein_features = np.zeros((len(pssm),20,13))

    translated_pssm = pssm.T
    offset_start=6
    #amino_class = []

    ix = 0
    for AA in range(0, len(pssm)):
        pssmmatrix = translated_pssm[:,AA+offset_start-6:AA+offset_start-6+13]
        pssmmatrix= shift(pssmmatrix, (0,offset_start), cval=0)
        if (len(pssmmatrix.T) <13):
            ncols = 13-len(pssmmatrix.T)
            aux = np.zeros((20,ncols), dtype="int8")
            pssmmatrix = np.hstack((pssmmatrix,aux))

        protein_features[ix] = pssmmatrix
        #amino_class.append(y_proteins[protein_index])
       
        if (offset_start >0):
            offset_start-=1
        ix+=1
   
    X_protein_features[protein] = protein_features
    protein_index +=1

In [6]:
print(len(proteins))
#remove proteins with wrong Structural information
for prot in proteins:
    if (len(X_protein_features[prot]) != len(protein_secundary_structure[prot])):
        protein_secundary_structure.pop(prot)
    if (len(X_protein_features[prot])<30):
        protein_secundary_structure.pop(prot)
        
proteins = list(protein_secundary_structure.keys())
print(len(proteins))


        

1461
1427


In [7]:
# for each proteins, for each amino extracts PSSM window with AA centered.
from scipy.ndimage import shift

X_protein_features = {}

protein_index = 0
for protein in proteins:
    pssm = protein_descritors[protein]
    protein_features = np.zeros((len(pssm),20,13))

    translated_pssm = pssm.T
    offset_start=6
    #amino_class = []

    ix = 0
    for AA in range(0, len(pssm)):
        pssmmatrix = translated_pssm[:,AA+offset_start-6:AA+offset_start-6+13]
        pssmmatrix= shift(pssmmatrix, (0,offset_start), cval=0)
        if (len(pssmmatrix.T) <13):
            ncols = 13-len(pssmmatrix.T)
            aux = np.zeros((20,ncols), dtype="int8")
            pssmmatrix = np.hstack((pssmmatrix,aux))

        protein_features[ix] = pssmmatrix
        #amino_class.append(y_proteins[protein_index])
       
        if (offset_start >0):
            offset_start-=1
        ix+=1
   
    X_protein_features[protein] = protein_features
    protein_index +=1

In [8]:
proteins = proteins[:100]

In [9]:
from sklearn.model_selection import train_test_split

proteins_train, proteins_test = train_test_split(proteins, test_size=0.30, random_state=42)


In [10]:
y_train = []
for prot_name in proteins_train:
    for S in protein_secundary_structure[prot_name]:
        y_train.append(S)
print(len(y_train))

y_test = []
for prot_name in proteins_test:
    for S in protein_secundary_structure[prot_name]:
        y_test.append(S)
print(len(y_test))

X_train = np.zeros((len(y_train), 20,13))
aa_ix = 0
for prot in proteins_train:
    for AA in X_protein_features[prot]:
        X_train[aa_ix] = AA
        aa_ix += 1

X_test = np.zeros((len(y_test), 20,13))
aa_ix = 0
for prot in proteins_test:
    for AA in X_protein_features[prot]:
        X_test[aa_ix] = AA
        aa_ix += 1
        
print(X_train.shape)
print(X_test.shape)

19296
4868
(19296, 20, 13)
(4868, 20, 13)


In [11]:
from sklearn import preprocessing
le = preprocessing.LabelEncoder()
le.fit(y_train)
y_train = le.transform(y_train)
y_test = le.transform(y_test)

In [12]:
import keras
INPUT_SHAPE = (20, 13, 1)  
inp = keras.layers.Input(shape=INPUT_SHAPE)
conv1 = keras.layers.Conv2D(500, kernel_size=(5, 5), activation='relu', padding='same')(inp)
pool1 = keras.layers.MaxPooling2D(pool_size=(2, 2))(conv1)
conv2 = keras.layers.Conv2D(100, kernel_size=(2, 2), activation='relu', padding='same')(pool1)
flat = keras.layers.Flatten()(conv2)  #Flatten the matrix to get it ready for dense.
hidden1 = keras.layers.Dense(50, activation='relu')(flat)
out = keras.layers.Dense(3, activation='softmax')(hidden1)   #units=1 gives error

modelcnn = keras.Model(inputs=inp, outputs=out)
modelcnn.compile(optimizer='adam',loss='categorical_crossentropy', metrics=['accuracy'])
print(modelcnn.summary())

Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_1 (InputLayer)        [(None, 20, 13, 1)]       0         
                                                                 
 conv2d (Conv2D)             (None, 20, 13, 500)       13000     
                                                                 
 max_pooling2d (MaxPooling2D  (None, 10, 6, 500)       0         
 )                                                               
                                                                 
 conv2d_1 (Conv2D)           (None, 10, 6, 100)        200100    
                                                                 
 flatten (Flatten)           (None, 6000)              0         
                                                                 
 dense (Dense)               (None, 50)                300050    
                                                             

In [13]:
from sklearn.model_selection import train_test_split
from keras.utils import to_categorical

history = modelcnn.fit(X_train,
                         to_categorical(np.array(list(y_train))),
                         batch_size = 128,
                         verbose = 1,
                         epochs = 5,      #Changed to 3 from 50 for testing purposes.
                         validation_split = 0.2,
                         shuffle = False
                      #   callbacks=callbacks
                   )

print("Test_Accuracy: {:.2f}%".format(modelcnn.evaluate(X_test, to_categorical(np.array(list(y_test))) )[1]*100))


Epoch 1/5
121/121 [==============================] - 15s 121ms/step - loss: 0.9447 - accuracy: 0.5796 - val_loss: 0.7015 - val_accuracy: 0.7047
Epoch 2/5
121/121 [==============================] - 14s 120ms/step - loss: 0.7577 - accuracy: 0.6827 - val_loss: 0.6592 - val_accuracy: 0.7272
Epoch 3/5
121/121 [==============================] - 14s 119ms/step - loss: 0.6927 - accuracy: 0.7160 - val_loss: 0.6453 - val_accuracy: 0.7420
Epoch 4/5
121/121 [==============================] - 14s 119ms/step - loss: 0.6194 - accuracy: 0.7392 - val_loss: 0.6607 - val_accuracy: 0.7342
Epoch 5/5
153/153 [==============================] - 1s 9ms/step - loss: 0.6613 - accuracy: 0.7529
Test_Accuracy: 75.29%


In [16]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM
from tensorflow.keras.layers import Dense, Flatten

modellstm = Sequential()
modellstm.add(LSTM(400, activation='relu', input_shape=(20, 13), return_sequences=True))
modellstm.add(LSTM(600, activation='relu', return_sequences=False))

modellstm.add(Flatten()) #Flatten the matrix to get it ready for dense.
modellstm.add(Dense(100, activation='relu'))
modellstm.add(Dense(3, activation='softmax'))

modellstm.compile(optimizer='adam',loss='categorical_crossentropy', metrics=['accuracy'])
modellstm.summary()


Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 lstm_2 (LSTM)               (None, 20, 400)           662400    
                                                                 
 lstm_3 (LSTM)               (None, 600)               2402400   
                                                                 
 flatten_2 (Flatten)         (None, 600)               0         
                                                                 
 dense_4 (Dense)             (None, 100)               60100     
                                                                 
 dense_5 (Dense)             (None, 3)                 303       
                                                                 
Total params: 3,125,203
Trainable params: 3,125,203
Non-trainable params: 0
_________________________________________________________________


In [17]:
from sklearn.model_selection import train_test_split
from keras.utils import to_categorical

history = modellstm.fit(X_train,
                         to_categorical(np.array(list(y_train))),
                         batch_size = 128,
                         verbose = 1,
                         epochs = 5,      #Changed to 3 from 50 for testing purposes.
                         validation_split = 0.2,
                         shuffle = False
                      #   callbacks=callbacks
                   )

print("Test_Accuracy: {:.2f}%".format(modellstm.evaluate(X_test, to_categorical(np.array(list(y_test))) )[1]*100))

Epoch 1/5
121/121 [==============================] - 39s 306ms/step - loss: 0.9900 - accuracy: 0.4967 - val_loss: 0.9874 - val_accuracy: 0.5119
Epoch 2/5
121/121 [==============================] - 36s 301ms/step - loss: 0.9749 - accuracy: 0.5250 - val_loss: 0.9517 - val_accuracy: 0.5119
Epoch 3/5
121/121 [==============================] - 37s 302ms/step - loss: 0.9401 - accuracy: 0.5320 - val_loss: 0.8975 - val_accuracy: 0.5119
Epoch 4/5
121/121 [==============================] - 38s 317ms/step - loss: 0.8937 - accuracy: 0.5707 - val_loss: 0.8411 - val_accuracy: 0.5637
Epoch 5/5
153/153 [==============================] - 7s 43ms/step - loss: 0.7665 - accuracy: 0.6559
Test_Accuracy: 65.59%


In [18]:
#ycnn = modelcnn.predict(X_train)
ytcnn = modelcnn.predict(X_test)

#ycnn = modelcnn.predict(X_train)
ytlstm = modellstm.predict(X_test)

ensembling = ((ytcnn*.42+ytlstm*.58)/2)

153/153 [==============================] - 7s 43ms/step


In [21]:
from sklearn import metrics
y_pred_f = []
for out in ensembling:
    y_pred_f.append(np.argmax(out))
    
print("Accuracy:",metrics.accuracy_score(y_test, y_pred_f))

Accuracy: 0.7524650780608052
